# 🚗 Automatic Number Plate Recognition (ANPR)
**Pipeline:** Plate Detection → Image Preprocessing → OCR → Annotated Output

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

## 1. Install Dependencies

In [ ]:
!pip install easyocr opencv-python-headless ultralytics matplotlib -q

## 2. Import Libraries

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src')

from detector import HaarPlateDetector
from ocr import PlateOCR, preprocess_plate
from pipeline import ANPRPipeline

print('All imports successful ✅')

## 3. Load a Test Image

In [ ]:
# Replace with your own image path
IMAGE_PATH = '../sample_images/car.jpg'

image = cv2.imread(IMAGE_PATH)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(image_rgb)
plt.title('Input Image')
plt.axis('off')
plt.show()

## 4. Stage 1 — Detect License Plate Region

In [ ]:
detector = HaarPlateDetector()
detections = detector.detect(image)

print(f'Plates detected: {len(detections)}')

for i, det in enumerate(detections):
    crop_rgb = cv2.cvtColor(det.crop, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6, 2))
    plt.imshow(crop_rgb)
    plt.title(f'Plate crop #{i+1}  |  bbox: {det.bbox}')
    plt.axis('off')
    plt.show()

## 5. Stage 2 — Preprocess + OCR

In [ ]:
ocr = PlateOCR(gpu=False)

for i, det in enumerate(detections):
    processed = preprocess_plate(det.crop)
    result = ocr.read(det.crop)

    fig, axes = plt.subplots(1, 2, figsize=(10, 2))
    axes[0].imshow(cv2.cvtColor(det.crop, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original Crop')
    axes[0].axis('off')

    axes[1].imshow(processed, cmap='gray')
    axes[1].set_title('After Preprocessing')
    axes[1].axis('off')

    plt.suptitle(f'Plate #{i+1}: "{result.plate_text if result else "No text"}"  (conf: {result.confidence:.0%})')
    plt.tight_layout()
    plt.show()

## 6. Full Pipeline — Annotated Output

In [ ]:
pipeline = ANPRPipeline(use_yolo=False)
results = pipeline.process_image(image)
annotated = pipeline.annotate(image, results)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title('ANPR — Annotated Output')
plt.axis('off')
plt.show()

print('\n🔍 Detected Plates:')
for r in results:
    print(f'  Plate: {r.plate_text:<15} | OCR conf: {r.ocr_conf:.0%} | Det conf: {r.detection_conf:.0%}')